# Task 5 ensemble: which book comes next?

Every test row asks for the book directly after the user's last training interaction (no timestamp in the test file).
Validation pairs are built the same way ("leave last out"): each user's last training interaction against the book of a
random other interaction (drawn in proportion to popularity). Sub-models score (user, book); a LightGBM pair blender decides.

- **popularity**, **item cosine**: as in Tasks 2 / 4
- **transition model**: which book follows which (Markov chain, PML Sec. 7.4)
- **EASE** and **time-weighted EASE**: one set of weights for the 50k most popular books (in RAM)
- **Mult-VAE**: non-linear autoencoder (GPU)
- **SASRec**: self-attention over the user's sequence (Kang & McAuley 2018, PML Sec. 7.7.2; GPU)

Every sub-model cell stores its scores in `trained_models/task5/<model>.pt`; the blend cells only need the stored files.
All hyperparameters: `submission/lib/config.py`, Task 5 section.

In [1]:
%load_ext autoreload
%autoreload 2

# assignment1.py and lib/ live in ./submission (this notebook lives in assignment1_2026)
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "submission"))

import torch

import assignment1
from lib.config import TASK5_PAIR_BLENDER
from lib.models.ease_model import load_ease_model, load_time_ease_model, store_ease_model, store_time_ease_model
from lib.models.item_cosine_model import load_item_cosine_model, store_item_cosine_model
from lib.models.mult_vae import load_mult_vae, store_mult_vae
from lib.models.pair_blender import load_pair_blender, store_pair_blender
from lib.models.popularity_model import load_popularity_model, store_popularity_model
from lib.models.sasrec import load_sasrec, store_sasrec
from lib.models.transition_model import load_transition_model, store_transition_model

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

2.14.1+cu130 | CUDA available: True | NVIDIA GeForce RTX 5080


### Data and validation pairs ("leave last out")
Each user's interactions in time order $i_{u,1}, \dots, i_{u,n}$. The test asks for the book right after $i_{u,n}$ (against a popularity-drawn negative).
Validation pairs: the positive is the user's last training interaction $i_{u,n}$, the history is $i_{u,1}, \dots, i_{u,n-1}$, the negative is the book of
a random other interaction ($\propto |U_i|$). "Now" $t$ = the day of the last history interaction (the test file has no timestamp).
$$\text{accuracy} = \frac{1}{|\mathcal P|}\sum_{(u,i^+,i^-)\in\mathcal P}\big[s(u,i^+) > s(u,i^-)\big]$$

In [2]:
# training file in chronological order -> leave-last-out validation pairs + the test pairs
task5 = assignment1.load_task5()

## Sub-models

### Popularity
$$s(u,i) = \log\big(1 + |U_i|\big)$$
$U_i$ = users who read $i$. On its own close to a coin flip here (the negatives are drawn in proportion to popularity), but the blender uses it to correct the popularity bias of the other scores.

In [3]:
# popularity: log(1 + #interactions of the book) (seconds)
popularity_result = assignment1.train_task5_popularity_model(task5)
store_popularity_model(popularity_result, "task5")

[03:55:25] popularity | validation accuracy 0.5204
[03:55:25] stored popularity -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task5\popularity.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task5/popularity.pt')

### Item cosine similarity to the user's history (PML Ch. 4)
$$\mathrm{sim}(i,j) = \frac{|U_i\cap U_j|}{\sqrt{|U_i|\,|U_j|}}\cdot\frac{n_{ij}}{n_{ij}+\lambda}, \qquad n_{ij} = |U_i\cap U_j|$$
$$s(u,i) = \sum_{j\in I_u\cap N_k(i)}\mathrm{sim}(i,j)$$
$N_k(i)$ = the $k = 100$ most similar books of $i$; $I_u$ = the books $u$ has read.

In [4]:
# item cosine similarity to the user's history (CPU, a few minutes: 232k books)
item_cosine_result = assignment1.train_task5_item_cosine_model(task5)
store_item_cosine_model(item_cosine_result, "task5")

[03:55:47] item cosine | validation accuracy 0.7228
[03:56:09] stored item_cosine -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task5\item_cosine.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task5/item_cosine.pt')

### Transition model: which book follows which (PML Sec. 7.4, Eq. 7.7)
$$C_{ji} = \sum_{u}\sum_{k}\sum_{\delta=1}^{D}\gamma^{\,\delta-1}\,\big[i_{u,k}=j,\ i_{u,k+\delta}=i\big], \qquad P(i\mid j) = \frac{C_{ji}}{\sum_{i'}C_{ji'}+\lambda}$$
$$s(u,i) = \sum_{m=1}^{M}\rho^{\,m-1}\,P\big(i\mid i_{u,n+1-m}\big)$$
$D = 3$ steps ahead, $\gamma = 0.5$, $\lambda = 5$; the user's last $M = 5$ books vote with weights $\rho^{m-1}$, $\rho = 0.5$.

In [5]:
# transition model: which book follows which, voted by the user's last books (CPU, seconds)
transition_result = assignment1.train_task5_transition_model(task5)
store_transition_model(transition_result, "task5")

[03:56:09] transition | validation accuracy 0.7019
[03:56:10] stored transition -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task5\transition.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task5/transition.pt')

### EASE: linear autoencoder in closed form (Steck 2019)
$X$ = user x book 0/1 matrix.
$$\hat B = \arg\min_B\ \lVert X - XB\rVert_F^2 + \lambda\lVert B\rVert_F^2 \quad \text{s.t.}\ \operatorname{diag}(B) = 0$$
$$P = \big(X^\top X + \lambda I\big)^{-1}, \qquad \hat B_{ij} = -\frac{P_{ij}}{P_{jj}}\ (i\neq j), \qquad \hat B_{jj} = 0$$
$$s(u,i) = \sum_{j\in I_u}\hat B_{ji} = \big(X\hat B\big)_{ui}$$
$\lambda = 500$; books outside the `max_items` most popular ones get no weights (score missing).

### Time-weighted EASE (same weights)
$$s(u,i,t) = \sum_{j\in I_u} e^{-|t - t_{uj}|/\tau}\,\hat B_{ji}, \qquad \tau = 180\ \text{days}, \quad t = \text{the user's last interaction}$$
Both variants come from one inverse per fit (50k books, in RAM).

In [6]:
# EASE + time-weighted EASE from one set of weights (50k books in RAM, ~20 GB peak, several minutes per fit)
ease_result, time_ease_result = assignment1.train_task5_ease_models(task5)
store_ease_model(ease_result, "task5")
store_time_ease_model(time_ease_result, "task5")

[03:58:25] EASE | validation accuracy 0.6885
[03:58:25] time-weighted EASE | validation accuracy 0.6784
[04:00:40] stored ease -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task5\ease.pt
[04:00:40] stored time_ease -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task5\time_ease.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task5/time_ease.pt')

### Mult-VAE: variational autoencoder (Liang et al. 2018)
$\mathbf x_u$ = the user's 0/1 history over the books. Encoder ($W_k$ = affine layers):
$$\big[\boldsymbol\mu_u,\ \log\boldsymbol\sigma_u^2\big] = W_2\tanh\Big(W_1\,\mathrm{dropout}\big(\mathbf x_u/\lVert\mathbf x_u\rVert\big)\Big), \qquad \mathbf z_u = \boldsymbol\mu_u + \boldsymbol\sigma_u\odot\boldsymbol\epsilon,\ \ \boldsymbol\epsilon\sim\mathcal N(0, I)$$
Decoder and loss (multinomial likelihood + KL term, $\beta$ raised linearly from 0 to 0.2 over 20 epochs):
$$\boldsymbol\pi_u = \operatorname{softmax}\big(W_4\tanh(W_3\,\mathbf z_u)\big), \qquad \mathcal L(u) = -\sum_i x_{ui}\log\pi_{ui} + \beta\,\mathrm{KL}\big(\mathcal N(\boldsymbol\mu_u, \boldsymbol\sigma_u^2)\,\Vert\,\mathcal N(0, I)\big)$$
Score: the logit with $\mathbf z_u = \boldsymbol\mu_u$, i.e. $s(u,i) = \big(W_4\tanh(W_3\boldsymbol\mu_u)\big)_i$. Sizes 600 / 200, dropout 0.5;
early stopping on the validation pair accuracy, then a refit on all interactions.

In [7]:
# Mult-VAE: early stopping on the validation pair accuracy, then refit on all interactions (GPU)
mult_vae_result = assignment1.train_task5_mult_vae(task5)
store_mult_vae(mult_vae_result, "task5")

[04:00:54] Mult-VAE epoch   1 | train loss 229.82 | validation accuracy 0.6098
[04:01:06] Mult-VAE epoch   2 | train loss 212.65 | validation accuracy 0.6598
[04:01:18] Mult-VAE epoch   3 | train loss 205.71 | validation accuracy 0.6775
[04:01:30] Mult-VAE epoch   4 | train loss 200.92 | validation accuracy 0.6906
[04:01:42] Mult-VAE epoch   5 | train loss 196.67 | validation accuracy 0.6959
[04:01:54] Mult-VAE epoch   6 | train loss 193.36 | validation accuracy 0.7038
[04:02:06] Mult-VAE epoch   7 | train loss 190.44 | validation accuracy 0.7066
[04:02:18] Mult-VAE epoch   8 | train loss 187.82 | validation accuracy 0.7065
[04:02:30] Mult-VAE epoch   9 | train loss 185.52 | validation accuracy 0.7089
[04:02:42] Mult-VAE epoch  10 | train loss 183.50 | validation accuracy 0.7095
[04:02:54] Mult-VAE epoch  11 | train loss 181.64 | validation accuracy 0.7098
[04:03:06] Mult-VAE epoch  12 | train loss 179.98 | validation accuracy 0.7104
[04:03:18] Mult-VAE epoch  13 | train loss 178.53 | 

WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task5/mult_vae.pt')

### SASRec: self-attentive sequential recommendation (Kang & McAuley 2018; PML Sec. 7.7.2)
Input: the user's last $L = 50$ books, item embeddings $\mathbf e_i$ and position embeddings $\mathbf p_\ell$:
$$\mathbf h^{(0)}_\ell = \sqrt d\,\mathbf e_{i_{u,\ell}} + \mathbf p_\ell, \qquad \mathbf h^{(b)} = \mathrm{FFN}\Big(\mathrm{LN}\big(\tilde{\mathbf h} + \mathrm{SelfAttn}_{\text{causal}}(\tilde{\mathbf h}, \mathbf h^{(b-1)}, \mathbf h^{(b-1)})\big)\Big),\ \ \tilde{\mathbf h} = \mathrm{LN}\big(\mathbf h^{(b-1)}\big)$$
Position $\ell$ only attends to positions $\le \ell$; $\mathbf h_\ell$ predicts the book at $\ell+1$ (shared embeddings):
$$\mathcal L = -\sum_{\ell}\Big[\log\sigma\big(\mathbf h_\ell^\top\mathbf e_{i_{u,\ell+1}}\big) + \log\Big(1-\sigma\big(\mathbf h_\ell^\top\mathbf e_{i^-}\big)\Big)\Big], \qquad i^- \propto |U_i|$$
$$s(u,i) = \mathbf h_L^\top\,\mathbf e_i$$
$d = 64$, 2 blocks, 1 head, dropout 0.5, Adam; early stopping on the validation pair accuracy, then a refit on all interactions.

In [8]:
# SASRec: early stopping on the validation pair accuracy, then refit on all interactions (GPU)
sasrec_result = assignment1.train_task5_sasrec(task5)
store_sasrec(sasrec_result, "task5")

[04:06:41] SASRec epoch   1 | train loss 1.2763 | validation accuracy 0.7959
[04:06:43] SASRec epoch   2 | train loss 1.0928 | validation accuracy 0.8282
[04:06:46] SASRec epoch   3 | train loss 0.9895 | validation accuracy 0.8506
[04:06:48] SASRec epoch   4 | train loss 0.9136 | validation accuracy 0.8596
[04:06:50] SASRec epoch   5 | train loss 0.8640 | validation accuracy 0.8647
[04:06:52] SASRec epoch   6 | train loss 0.8277 | validation accuracy 0.8701
[04:06:54] SASRec epoch   7 | train loss 0.7948 | validation accuracy 0.8740
[04:06:57] SASRec epoch   8 | train loss 0.7694 | validation accuracy 0.8770
[04:06:59] SASRec epoch   9 | train loss 0.7487 | validation accuracy 0.8787
[04:07:01] SASRec epoch  10 | train loss 0.7285 | validation accuracy 0.8812
[04:07:03] SASRec epoch  11 | train loss 0.7125 | validation accuracy 0.8811
[04:07:06] SASRec epoch  12 | train loss 0.6999 | validation accuracy 0.8815
[04:07:08] SASRec epoch  13 | train loss 0.6867 | validation accuracy 0.8825

WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task5/sasrec.pt')

## Blend

### Blender input
For every validation pair: the scores $s^{(m)}(u, i_1)$, $s^{(m)}(u, i_2)$ of both books from every sub-model in `combined_models` (models that never saw the held-out interaction).

In [9]:
# stored sub-models of the blend (also works right after a kernel restart)
sub_model_loaders = {
    "popularity": load_popularity_model,
    "item_cosine": load_item_cosine_model,
    "transition": load_transition_model,
    "ease": load_ease_model,
    "time_ease": load_time_ease_model,
    "mult_vae": load_mult_vae,
    "sasrec": load_sasrec,
}
task5_model_results = {model_name: sub_model_loaders[model_name]("task5") for model_name in TASK5_PAIR_BLENDER.combined_models}

### Pair blender (LightGBM classifier)
Features of book $k$ of a pair: $\mathbf f_k = \big[s^{(1)}(u,i_k), \dots, s^{(M)}(u,i_k),\ [i_k \in I_u]\big]$ (+ time features where available). Features of the pair in a given order:
$$\mathbf x_{12} = \big[\mathbf f_1 - \mathbf f_2,\ \mathbf f_1,\ \mathbf f_2,\ \log(1+|I_u|)\big]$$
$g$ = gradient-boosted trees with log loss, trained on every validation pair in both orders ($\mathbf x_{12}$ with label $y$, $\mathbf x_{21}$ with $1-y$):
$$P(i_1\ \text{is the real book}) = \tfrac12\Big(g(\mathbf x_{12}) + 1 - g(\mathbf x_{21})\Big)$$
Score: 5-fold out-of-fold accuracy on the validation pairs. Prediction: item1 if the probability is above 0.5.

Time features of book $k$: $\log\big(1 + n_{i_k}^{\pm 30\text{ days}}(t)\big)$ (interactions with the book around $t$) and the book's age at $t$ in years
(negative = first read by anyone only after $t$).

In [10]:
# LightGBM pair blender: P(item1 is the next book) from both books' scores + meta-features (prints the validation accuracies)
pair_blend_result = assignment1.train_task5_blender(task5, task5_model_results)
store_pair_blender(pair_blend_result, "task5")

[04:08:51] popularity             validation accuracy 0.5204
[04:08:51] item_cosine            validation accuracy 0.7228
[04:08:51] transition             validation accuracy 0.7019
[04:08:51] ease                   validation accuracy 0.6885
[04:08:51] time_ease              validation accuracy 0.6784
[04:08:51] mult_vae               validation accuracy 0.7104
[04:08:51] sasrec                 validation accuracy 0.8854
[04:08:51] pair blend             validation accuracy 0.9205  (out-of-fold, 5 folds)
[04:08:52] pair blend inputs: sasrec_difference 858, item_age_years_at_t_first 738, popularity_second 727, item_age_years_at_t_second 726, popularity_first 703, mult_vae_difference 696, popularity_difference 619, mult_vae_first 469, sasrec_second 457, mult_vae_second 442, sasrec_first 440, time_ease_second 412, ease_first 404, ease_second 387, time_ease_first 381, log_user_interactions 373, transition_difference 337, item_age_years_at_t_difference 316, log_interactions_around_t_diffe

WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task5/pair_blender.pt')

### Submission
`which` = item1 where $P(i_1\ \text{is the real book}) > 0.5$, otherwise item2.

In [11]:
# writes submission/output/task_5_predictions.csv (the "which" column = the chosen book)
submitted_blend_result = load_pair_blender("task5")
assignment1.write_task5_predictions(submitted_blend_result.test_predictions)
print("wrote", assignment1.TASK5_PREDICTION_FILE)

wrote C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\submission\output\task_5_predictions.csv
